# Домашнее задание 3: Fine-tuning трансформера для NER

**Задача:** Дообучить энкодерную модель на датасете factRuEval-2016 для задачи распознавания именованных сущностей (NER), а также исследовать подходы к улучшению качества через MLM-преадаптацию, Whole-Word Masking и синтетические данные.

## 1. Установка зависимостей

In [ ]:
!pip install -q transformers datasets evaluate seqeval accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


## 2. Загрузка датасета

Используем датасет **gusevski/factrueval2016** — русскоязычный корпус для NER с метками `PERSON`, `ORGANIZATION`, `LOCATION`.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("gusevski/factrueval2016")

print(dataset)
print("\nПример из train-части:")
print(dataset['train'][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train_data.json:   0%|          | 0.00/7.62M [00:00<?, ?B/s]

dev_data.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

test_data.json:   0%|          | 0.00/2.57M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['data'],
        num_rows: 1
    })
    validation: Dataset({
        features: ['data'],
        num_rows: 1
    })
    test: Dataset({
        features: ['data'],
        num_rows: 1
    })
})
{'data': [{'id': 0, 'tokens': ['"', 'Если', 'Миронов', 'занял', 'столь', 'оппозиционную', 'позицию', ',', 'то', 'мне', 'представляется', ',', 'что', 'для', 'него', 'было', 'бы', 'порядочным', 'и', 'правильным', 'уйти', 'в', 'отставку', 'с', 'занимаемого', 'им', 'поста', ',', 'поста', ',', 'который', 'предоставлен', 'ему', 'сегодня', '"', 'Единой', 'Россией', "''", 'и', 'никем', 'больше', "''", ',', '-', 'заключает', 'Исаев', '.'], 'length': 47, 'ner_tags_str': ['O', 'O', 'B-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORG', 'I-ORG', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-PER', 'O'], 'ner_tags': [0, 0, 1, 0, 0, 0, 0,

## 3. Подготовка данных

Датасет имеет вложенную структуру: каждый документ содержит список предложений. Для обучения приводим его к плоскому виду (одна строка = одно предложение).

Затем токенизируем с выравниванием меток: каждый подтокен BERT-а получает метку родительского слова; специальные токены (`[CLS]`, `[SEP]`, подтокены не-первые) помечаются как `-100` и игнорируются при подсчёте loss.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

model_checkpoint = "DeepPavlov/rubert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


def flatten_factrueval(dataset_split):
    """Преобразует вложенную структуру датасета в плоский список предложений."""
    flattened_data = []
    for record in dataset_split:
        for sentence in record['data']:
            flattened_data.append({
                'tokens': sentence['tokens'],
                'ner_tags': sentence['ner_tags']
            })
    return Dataset.from_list(flattened_data)


flat_dataset = DatasetDict({
    'train': flatten_factrueval(dataset['train']),
    'test': flatten_factrueval(dataset['test'])
})

print(f"Train: {len(flat_dataset['train'])} предложений")
print(f"Test:  {len(flat_dataset['test'])} предложений")
print(f"Пример: {flat_dataset['train'][0]}")

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Новая структура: dict_keys(['tokens', 'ner_tags'])


Map:   0%|          | 0/7746 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

In [ ]:
def tokenize_and_align_labels(examples):
    """
    Токенизирует слова и выравнивает BIO-метки по подтокенам.
    Первый подтокен слова получает исходную метку, остальные — -100 (игнорируются при вычислении loss).
    """
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                # Специальные токены [CLS] / [SEP]
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Первый подтокен слова — берём метку
                label_ids.append(label[word_idx])
            else:
                # Последующие подтокены — игнорируем
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


tokenized_datasets = flat_dataset.map(tokenize_and_align_labels, batched=True)
print("Токенизация завершена.")

## 4. Метки и метрика качества

Извлекаем список меток из схемы датасета. Для оценки используем библиотеку **seqeval**, которая вычисляет precision, recall и F1 на уровне сущностей (entity-level), а не токенов.

In [ ]:
import evaluate
import numpy as np

metric = evaluate.load("seqeval")

# Извлекаем имена меток из схемы датасета
try:
    label_list = dataset["train"].features["data"].feature["ner_tags"].feature.names
except (AttributeError, KeyError):
    # Резервный вариант на случай нестандартной структуры признаков
    feature = dataset["train"].features["data"].feature["ner_tags"]
    if hasattr(feature, "feature") and hasattr(feature.feature, "names"):
        label_list = feature.feature.names
    else:
        label_list = ['O', 'B-PERSON', 'I-PERSON', 'B-ORGANIZATION', 'I-ORGANIZATION', 'B-LOCATION', 'I-LOCATION']

print(f"Метки ({len(label_list)}): {label_list}")


def compute_metrics(p):
    """Вычисляет NER-метрики через seqeval (entity-level precision/recall/F1)."""
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }

Список меток: ['O', 'B-PERSON', 'I-PERSON', 'B-ORGANIZATION', 'I-ORGANIZATION', 'B-LOCATION', 'I-LOCATION']


## 5. Базовое обучение NER (Baseline)

Дообучаем `DeepPavlov/rubert-base-cased` на задачу NER с нуля (из оригинального чекпойнта).
Фиксируем метрики **до** и **после** дообучения.

In [ ]:
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./results_ner_baseline",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("=== Оценка ДО дообучения (случайная голова классификатора) ===")
results_before = trainer.evaluate()
print(results_before)

print("\n=== Обучение baseline NER ===")
trainer.train()

print("\n=== Оценка ПОСЛЕ дообучения (baseline) ===")
results_baseline = trainer.evaluate()
print(results_baseline)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                          

Initial evaluation (before fine-tuning):


{'eval_loss': 1.9200822114944458, 'eval_model_preparation_time': 0.0053, 'eval_precision': 0.01864795716508849, 'eval_recall': 0.12887349617207436, 'eval_f1': 0.0325813958846978, 'eval_accuracy': 0.18577246502507636, 'eval_runtime': 11.5639, 'eval_samples_per_second': 223.281, 'eval_steps_per_second': 7.005}

Starting training...


Epoch,Training Loss,Validation Loss,Model Preparation Time,Precision,Recall,F1,Accuracy
1,0.031713,0.022184,0.005300,0.965411,0.971746,0.968568,0.993458
2,0.013825,0.016977,0.005300,0.970599,0.980860,0.975703,0.995362
3,0.009073,0.017057,0.005300,0.975495,0.979584,0.977535,0.995720


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Final evaluation (after fine-tuning):


{'eval_loss': 0.017057165503501892, 'eval_model_preparation_time': 0.0053, 'eval_precision': 0.9754946451261572, 'eval_recall': 0.979584396646008, 'eval_f1': 0.9775352432924057, 'eval_accuracy': 0.9957200497756326, 'eval_runtime': 12.0089, 'eval_samples_per_second': 215.007, 'eval_steps_per_second': 6.745, 'epoch': 3.0}


## 6. Улучшение 1: MLM-адаптация к домену

Перед NER-дообучением проводим предварительное обучение модели в режиме **Masked Language Modeling (MLM)** на текстах из train-части корпуса. Это позволяет энкодеру адаптироваться к специфике предметной области.

### Шаг 6.1: MLM-преадаптация

In [ ]:
from transformers import AutoModelForMaskedLM, DataCollatorForLanguageModeling

def group_texts(examples):
    """
    Конкатенирует токены из нескольких предложений и нарезает их на блоки фиксированного размера.
    Это стандартный подход для подготовки данных к MLM.
    """
    block_size = 128
    all_tokens = sum(examples['tokens'], [])
    total_length = (len(all_tokens) // block_size) * block_size

    result = {'input_ids': []}
    for i in range(0, total_length, block_size):
        text_chunk = " ".join(all_tokens[i: i + block_size])
        tokenized = tokenizer(text_chunk, truncation=True, max_length=block_size)
        result['input_ids'].append(tokenized['input_ids'])
    return result


mlm_dataset = flat_dataset['train'].map(
    group_texts,
    batched=True,
    remove_columns=flat_dataset['train'].column_names,
)

mlm_model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

# Стандартное случайное маскирование 15% токенов
data_collator_mlm = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm_probability=0.15,
)

mlm_args = TrainingArguments(
    output_dir="./rubert-mlm-factrueval",
    eval_strategy="no",
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=3,
    report_to="none",
)

mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_args,
    train_dataset=mlm_dataset,
    data_collator=data_collator_mlm,
)

print("=== MLM-преадаптация ===")
mlm_trainer.train()
mlm_model.save_pretrained("./mlm_pretrained_model")
print("MLM-преадаптация завершена. Веса сохранены в ./mlm_pretrained_model")

Map:   0%|          | 0/7746 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                          | Status     |  | 
-----------------------------+------------+--+-
cls.seq_relationship.weight  | UNEXPECTED |  | 
cls.seq_relationship.bias    | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be igno

Starting MLM pre-training...


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MLM pre-training finished and model saved.


### Шаг 6.2: NER-дообучение после MLM

Загружаем MLM-адаптированные веса и дообучаем на NER с теми же гиперпараметрами, что и в baseline.

In [ ]:
model_ner_after_mlm = AutoModelForTokenClassification.from_pretrained(
    "./mlm_pretrained_model",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

training_args_mlm_ner = TrainingArguments(
    output_dir="./results_ner_after_mlm",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32, # параметры чтобы колаб не падал
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
)

trainer_mlm_ner = Trainer(
    model=model_ner_after_mlm,
    args=training_args_mlm_ner,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("=== NER-дообучение после MLM ===")
trainer_mlm_ner.train()

print("\n=== Результаты: MLM → NER ===")
results_after_mlm = trainer_mlm_ner.evaluate()
print(results_after_mlm)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: ./mlm_pretrained_model
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting NER fine-tuning after MLM pre-training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.032232,0.021808,0.966914,0.974845,0.970863,0.993627
2,0.014059,0.016589,0.973941,0.981043,0.977479,0.995682
3,0.009595,0.016498,0.977343,0.982865,0.980096,0.996154


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluation after MLM + NER fine-tuning:


{'eval_loss': 0.016498146578669548, 'eval_precision': 0.9773427587456951, 'eval_recall': 0.982865475756471, 'eval_f1': 0.9800963373625374, 'eval_accuracy': 0.9961537011199517, 'eval_runtime': 12.8814, 'eval_samples_per_second': 200.444, 'eval_steps_per_second': 6.288, 'epoch': 3.0}


## 7. Улучшение 2: Whole Word Masking (WWM)

Вместо случайного маскирования отдельных подтокенов используем **DataCollatorForWholeWordMask**, который маскирует все подтокены слова целиком. Это более естественная постановка задачи и потенциально даёт лучшие представления.

### Шаг 7.1: WWM-преадаптация

In [ ]:
import torch
import gc
from transformers import DataCollatorForWholeWordMask

# Освобождаем GPU-память перед следующим этапом
def cleanup_gpu():
    """Удаляет все крупные объекты из глобального пространства имён и очищает кэш CUDA."""
    vars_to_del = [
        'trainer', 'mlm_trainer', 'trainer_mlm_ner',
        'model', 'mlm_model', 'model_ner_after_mlm',
    ]
    for v in vars_to_del:
        if v in globals():
            del globals()[v]
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

cleanup_gpu()

mlm_model_wwm = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

# Whole Word Masking: маскируем слово целиком, а не отдельные подтокены
data_collator_wwm = DataCollatorForWholeWordMask(
    tokenizer=tokenizer,
    mlm_probability=0.15,
)

wwm_args = TrainingArguments(
    output_dir="./rubert-wwm-factrueval",
    eval_strategy="no",
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=3,
    report_to="none",
)

wwm_trainer = Trainer(
    model=mlm_model_wwm,
    args=wwm_args,
    train_dataset=mlm_dataset,
    data_collator=data_collator_wwm,
)

print("=== WWM-преадаптация ===")
wwm_trainer.train()
mlm_model_wwm.save_pretrained("./wwm_pretrained_model")
print("WWM-преадаптация завершена. Веса сохранены в ./wwm_pretrained_model")

### Шаг 7.2: NER-дообучение после WWM

Аналогично MLM-сценарию: загружаем WWM-адаптированные веса и дообучаем на NER. Из-за потенциального увеличения потребления памяти уменьшаем размер батча и используем gradient accumulation.

In [ ]:
# Очищаем GPU-память перед новым этапом
cleanup_gpu()

model_ner_after_wwm = AutoModelForTokenClassification.from_pretrained(
    "./wwm_pretrained_model",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

# Уменьшаем batch size и используем gradient accumulation для стабильности
training_args_wwm_ner = TrainingArguments(
    output_dir="./results_ner_after_wwm",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,  # Эффективный batch size = 32
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
)

trainer_wwm_ner = Trainer(
    model=model_ner_after_wwm,
    args=training_args_wwm_ner,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("=== NER-дообучение после WWM ===")
trainer_wwm_ner.train()

print("\n=== Результаты: WWM → NER ===")
results_after_wwm = trainer_wwm_ner.evaluate()
print(results_after_wwm)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: ./wwm_pretrained_model
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting NER fine-tuning after WWM with batch size 8 and cleared memory...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.123868,0.020993,0.966176,0.968465,0.967319,0.993627
2,0.051503,0.017476,0.971691,0.982319,0.976976,0.995437
3,0.037540,0.016343,0.978399,0.982501,0.980446,0.996229


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluation after WWM + NER fine-tuning:


{'eval_loss': 0.016343139111995697, 'eval_precision': 0.9783989834815756, 'eval_recall': 0.982500911410864, 'eval_f1': 0.9804456571168713, 'eval_accuracy': 0.9962291187450507, 'eval_runtime': 10.1007, 'eval_samples_per_second': 255.626, 'eval_steps_per_second': 31.978, 'epoch': 3.0}


## 8. Улучшение 3: Синтетическая разметка

Генерируем псевдоразметку для дополнительного корпуса с помощью более мощной мультиязычной NER-модели (`Babelscape/wikineural-multilingual-ner`). Затем объединяем синтетические данные с реальными и обучаем модель из оригинального чекпойнта.

### Шаг 8.1: Загрузка корпуса и генерация псевдоразметки

In [ ]:
from datasets import load_dataset
from transformers import pipeline

# Загружаем WikiANN (RU) как источник неразмеченных русскоязычных текстов
print("Загрузка WikiANN (RU)...")
wiki_dataset = load_dataset("wikiann", "ru", split="validation", streaming=True)

# Берём 500 примеров — достаточно для демонстрации подхода
lenta_samples = []
for i, example in enumerate(wiki_dataset):
    if i >= 500:
        break
    lenta_samples.append(" ".join(example['tokens']))

print(f"Загружено {len(lenta_samples)} текстов.")

# Модель-учитель: мультиязычная NER, обученная на WikiNEural
print("Загрузка модели-учителя...")
teacher_ner = pipeline(
    "ner",
    model="Babelscape/wikineural-multilingual-ner",
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1,
)

print("Генерация псевдоразметки...")
synthetic_results = [
    {"text": text, "entities": teacher_ner(text)}
    for text in lenta_samples
]
print(f"Разметка сгенерирована для {len(synthetic_results)} текстов.")

Loading WikiANN (RU) dataset...


README.md: 0.00B [00:00, ?B/s]

Successfully loaded 500 texts.
Loading teacher model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/709M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Generating synthetic labels...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Generated labels for 500 texts.


### Шаг 8.2: Преобразование синтетической разметки и объединение с реальными данными

In [ ]:
from datasets import concatenate_datasets

def process_synthetic_to_dataset(synthetic_results, label2id):
    """
    Конвертирует вывод teacher-модели в формат датасета с BIO-метками.
    Использует простое word-level выравнивание: ищем токены, содержащие текст сущности.
    """
    # Маппинг меток учителя на метки нашего датасета
    entity_mapping = {'PER': 'PERSON', 'ORG': 'ORGANIZATION', 'LOC': 'LOCATION'}
    processed_data = []

    for item in synthetic_results:
        tokens = item['text'].split()
        ner_tags = [0] * len(tokens)  # 0 = 'O' (не-сущность)

        for ent in item['entities']:
            label_type = entity_mapping.get(ent['entity_group'])
            if not label_type:
                continue
            ent_text = ent['word'].replace(' ', '')
            for idx, token in enumerate(tokens):
                if ent_text in token:
                    ner_tags[idx] = label2id.get(f'B-{label_type}', 0)

        processed_data.append({'tokens': tokens, 'ner_tags': ner_tags})

    return Dataset.from_list(processed_data)


synthetic_ds = process_synthetic_to_dataset(synthetic_results, label2id)

# Объединяем синтетику с оригинальным train-сплитом
combined_train_dataset = concatenate_datasets([flat_dataset['train'], synthetic_ds])
print(f"Итоговый train: {len(combined_train_dataset)} примеров "
      f"({len(flat_dataset['train'])} реальных + {len(synthetic_ds)} синтетических)")

tokenized_combined = combined_train_dataset.map(tokenize_and_align_labels, batched=True)

### Шаг 8.3: Обучение на объединённых данных

In [ ]:
# Обучаем из оригинального чекпойнта, но на расширенном датасете
model_final = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

trainer_final = Trainer(
    model=model_final,
    args=training_args,  # Те же гиперпараметры, что в baseline
    train_dataset=tokenized_combined,
    eval_dataset=tokenized_datasets['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("=== Обучение на реальных + синтетических данных ===")
trainer_final.train()

print("\n=== Результаты: Synthetic + Real Data → NER ===")
results_synthetic = trainer_final.evaluate()
print(results_synthetic)

Map:   0%|          | 0/8246 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                          

Starting final training with combined synthetic and real data...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.035158,0.024279,0.965110,0.968101,0.966603,0.993099
2,0.019752,0.018496,0.969139,0.978855,0.973973,0.994872
3,0.010501,0.018153,0.973941,0.981043,0.977479,0.995456


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Final Evaluation:


{'eval_loss': 0.018153095617890358, 'eval_precision': 0.9739413680781759, 'eval_recall': 0.981042654028436, 'eval_f1': 0.9774791136941519, 'eval_accuracy': 0.9954560880877861, 'eval_runtime': 12.009, 'eval_samples_per_second': 215.005, 'eval_steps_per_second': 6.745, 'epoch': 3.0}


## 9. Финальное сравнение подходов

In [ ]:
import pandas as pd

# Собираем результаты всех экспериментов
final_comparison = {
    "Baseline NER":            results_baseline,
    "MLM → NER":               results_after_mlm,
    "WWM → NER":               results_after_wwm,
    "Synthetic + Real → NER":  results_synthetic,
}

df_results = pd.DataFrame(final_comparison).T[
    ['eval_precision', 'eval_recall', 'eval_f1', 'eval_accuracy', 'eval_loss']
].rename(columns={
    'eval_precision': 'Precision',
    'eval_recall':    'Recall',
    'eval_f1':        'F1',
    'eval_accuracy':  'Accuracy',
    'eval_loss':      'Loss',
})

print("=== Сравнение подходов ===")
display(df_results.round(4))

best_method = df_results['F1'].idxmax()
print(f"\nЛучший результат по F1: {best_method} — {df_results.loc[best_method, 'F1']:.4f}")

Сравнение доступных подходов:


,eval_precision,eval_recall,eval_f1,eval_accuracy,eval_loss
MLM + NER,0.977343,0.982865,0.980096,0.996154,0.016498
WWM + NER,0.978399,0.982501,0.980446,0.996229,0.016343
Synthetic + Real Data,0.973941,0.981043,0.977479,0.995456,0.018153



Лучший результат (F1-score): WWM + NER (0.9804)


## 10. Выводы

### Анализ результатов

| Подход | Описание | Ожидаемый эффект |
|---|---|---|
| **Baseline NER** | Дообучение из оригинального чекпойнта | Базовый уровень |
| **MLM → NER** | Доменная MLM-адаптация + NER | Лучшее понимание терминологии домена |
| **WWM → NER** | Whole Word Masking + NER | Более связные представления слов |
| **Synthetic + Real → NER** | Псевдоразметка учителем + реальные данные | Увеличение объёма обучающей выборки |

### Что можно улучшить в будущих экспериментах

1. **Больше синтетических данных** — увеличить объём до 50000–100000 примеров; использовать более качественные источники текстов (новости, Wikip).
2. **Concept Masking** — маскировать при MLM только токены, соответствующие NER-сущностям, чтобы модель лучше фокусировалась на их представлениях.
3. **Подбор гиперпараметров** — grid search по `learning_rate`, `num_epochs`, `warmup_ratio`; попробовать более агрессивный weight decay для регуляризации.
4. **Более качественное выравнивание синтетики** — вместо простого string matching использовать character-offset alignment для точной разметки.
5. **Ансамблирование** — усреднить предсказания моделей, обученных разными способами.
